# The Missing Curriculum: Essential Concepts For Data Scientists in the Age of AI Coding Agents

AI can write the code, but you have to steer the ship. Master the knowledge to keep you relevant in the age of AI.

## Why read this article?

This article isn't yet another one about how to structure your prompts to enable your AI agent to perform magic. There are already a sea of articles that goes into detail about what structure to use and when so there's no need for another. 

Instead, this article is one out of a series of articles that are about how to keep yourself, the coder, relevant in the modern AI coding ecosystem. 

> It's about learning the techniques that enable you to excel in utilising coding agents better then those who blindly hit tab or copy-paste.

We will go into the concepts from existing software engineering practices that you should be aware of, and go into why these concepts are relevant, particularly now.

- By reading this series, you should have a good idea of what common pitfalls to look for in auto-generated code, and know how to guide a coding assistant to create production grade code that is maintainable and extensible.

- I'd say this article is most relevant for budding programmers, graduates, and professionals from other technical industries that want to level up their coding expertise.

What we will cover not only makes you better at using coding assistants but also better coders in general.

## The Core Concepts

The high level concepts we'll cover in this series are the following:

- Code Smells
- Design Patterns
- Abstraction

In essence, there's nothing new about them. To seasoned developers, they are second nature, drilled into their brains through years of PR reviews and debugging. You eventually reach a point where you instinctively react to code that "feels" like future pain.

And now, they are perhaps more relevant than ever since coding assistants have become an essential part of any developers' experience, be it juniors to seniors.

Why? 

Because the manual labor of writing code has been offloaded. The primary responsibility for any developer has shifted from writing code to reviewing it. Everyone has effectively become a senior developer guiding a junior (the coding assistant).

So, it's become essential for even junior software practitioners to be able to 'review' code, but with the foresight of a senior developer. This is why we will be covering the above concepts so that in the very very least, you can tell your coding assistant to take them into account, even if you yourself don't exactly know what you're looking for.

So, introductions are now done. Let's get straight into our first topic: Code smells.

## Code Smells

What is a code smell? I find it a very aptly named term - it's the equivalent of sour smelling milk indicating to you that it's a bad idea to drink it.

For decades, developers have learnt through trial and error what kind of code works long-term. "Smelly" code are brittle, prone to hidden bugs, and difficult for a human or AI agent to understand exactly what's going on. Thus it is very useful for developers to know about code smells and how to detect them.

Useful links: 
- https://luzkan.github.io/smells/
- https://refactoring.guru/refactoring/smells

Now, having used coding agents to build everything from professional ML pipelines for my 9-5 job to entire mobile apps in languages I’d never touched before for my side-projects, I've identified two typical "smells" that emerges when you become over-reliant on just the AI output:

- Divergent Change
- Speculative Generality

Let's go through what they are, the risks involved, and examples on how to fix them.

### Divergent Change

Divergent change is when a single module or class is doing too many things at once. The purpose of the code 'diverges' into many different directions and so rather than being focused on being good at one task (Single Responsibility Principle), it is trying to do everything.

This results in a painful situation where this code is always breaking, and thus requires fixing for various independent reasons.

#### When does it happen with AI? 

When the developer is not engaged with the codebase and blindly accepts the Agent output, you are doubly susceptible to this. 

Yes, you may have done all the correct things and made a nicely structured prompt that adheres to the latest in prompt engineering. But if you ask it to "add functionality to handle X," the agent will usually do exactly as it is told and cram code into your existing class, especially when the existing codebase is already very complicated.

It is ultimately up to you to take into account the role, responsiblity and intended usage of the code to come up with a holistic approach. Otherwise, you're very likely to end up with smelly code.

### Example - ML Engineering

Below, we have a `ModelPipeline` class from which you can get whiffs of future extensibility issues.

A quick warning: We can't talk in absolutes and say this code is bad just for the sake of it.

It always depends on the wider context of how code is used. For a simple codebase that isn't expected to grow in scope, the below is perfectly fine.

Also, note that it's a contrived and simple example to illustrate the concept. Don't bother giving this to an agent to prove it can figure out this is smelly without being told so. The point is for *you* to recognise the smell before the agent makes it worse.

```python
class ModelPipeline:
    def __init__(self, data_path):
        self.data_path = data_path

    def load_from_s3(self):
        print(f"Connecting to S3 to get {self.data_path}")
        return "raw_data"

    def clean_txn_data(self, data):
        print("Cleaning specific transaction JSON format")
        return "cleaned_data"

    def train_xgboost(self, data):
        print("Running XGBoost trainer")
        return "model"
```

So, what are things that should be going through your head when you look at this code?

- Data retrieval: What happens when we start having more than one data source, like Bigquery tables, local databases, or Azure blobs? How likely is this to happen? 
- Data Engineering: If the upstream data changes or downstream modelling changes, this will also need to change.
- Modelling: If we use different models, LightGBM or some Neural Net, the upstream modelling needs to change.

You should notice that this class couples Platform, Data engineering, and ML engineering concerns into a single place. Thus, we've tripled the reason for this code to be modified - i.e. code that is beginning to smell like 'divergent change'.

#### Why is this a 'possible' problem?

1. Operational risk: Every edit runs the risk of introducing a bug, be it human or AI. By having this class wear three different hats, you've tripled the risk of this breaking, since there's three times as more reasons for this code to change.

2. AI Agent Context Pollution: The Agent sees the cleaning and training code as part of the same problem. For example, it is more likely to change the training and data loading logic to accommodate a change in the data engineering, even though it was unnecessary. Ultimately, this increases the 'divergent change' code smell.

3. Risk is magnified by AI: An agent can rewrite hundreds of lines of code in a second. If those lines represent three different disciplines, the agent has just tripled the chance of introducing a bug that your unit tests or your PR reviewer might not catch.

### How to fix it?

The risks outlined above should give you some ideas about how to refactor this code.

One possible approach is as below:

```python
class S3DataLoader:
    """Handles only Infrastructure concerns."""
    def __init__(self, data_path):
        self.data_path = data_path

    def load(self):
        print(f"Connecting to S3 to get {self.data_path}")
        return "raw_data"

class TransactionsCleaner:
    """Handles only Data Domain/Schema concerns."""
    def clean(self, data):
        print("Cleaning specific transaction JSON format")
        return "cleaned_data"

class XGBoostTrainer:
    """Handles only ML/Research concerns."""
    def train(self, data):
        print("Running XGBoost trainer")
        return "model"

class ModelPipeline:
    """The Orchestrator: It knows 'what' to do, but not 'how' to do it."""
    def __init__(self, loader, cleaner, trainer):
        self.loader = loader
        self.cleaner = cleaner
        self.trainer = trainer

    def run(self):
        data = self.loader.load()
        cleaned = self.cleaner.clean(data)
        return self.trainer.train(cleaned)
```

Formerly, the model pipeline's responsibility was to handle everything. Now, it's responsibility is to orchestrate different modelling stages, whilst the complexities of each stage is cleanly separated into their own respective classes.

What does this achieve?

1. Minimized Operational Risk: Now, concerns are decoupled and responsibilities are stark clear. You can refactor your data loading logic with confidence that the ML training code remains untouched. As long as the inputs and outputs (the "contracts") stay the same, the risk of impacting anything downstream is lowered.

2. Testable Isolation: It is significantly easier to write unit tests since the scope of testing is smaller and well defined for each individual class.

3. Lego-brick Flexibility: The architecture is now open for extension. Need to migrate from S3 to Azure? Simply drop in an AzureBlobLoader. Want to experiment with LightGBM? Swap the trainer.

You ultimately end up with code that is more reliable, readable, and maintainable for both you and the AI agent.

## Speculative Generality

Whilst 'Divergent Change' occurs most often in an already large and complicated codebase, 'Speculative Generality' seems to occur when you start out creating a new project.

This code smell is when the developer tries to futureproof a project by guessing how things will pan out, resulting in unnecessary functionality that only increases complexity.

We've all been there:

> "I'll make this model training pipeline support all kinds of models, cross validation and hyperparameter tuning methods, and make sure there's human-in-the-loop feedback for model selection so that we can use this for all of our training in the future"

only to find that it's a monster of a job, code turns out flaky, you spend too much time on it whilst you've not been able to build out the simple LightGBM classification model that you needed in the first place.

### When AI Agents are susceptible to this smell

I've found that the latest, high performing coding agents are most susceptible to this smell. Couple the powerful agent with a vague prompt, and you quickly end up with too many modules and hundreds of lines of new code.

Perhaps every line is pure gold and it's exactly what you need. When I experienced something like this recently, the code certainly seemed to make sense to me.

But I ended up rejecting all of it. Why?

Because the agent was making design choices for a future I hadn't even mapped out yet.

### The Key Principle: Grow your codebase organically

The mantra to remember when reviewing AI output is "YAGNI" (You ain't gonna need it). It's a principle in software development that suggests you should only implement the code you need, not the code you foresee.

Start with the simplest thing that works. Then, iterate on it.

This is a more natural, organic way of growing your codebase that gets things done, whilst also being lean, simple, and less susceptible to bugs.

### Revisiting our examples

We previously looked at refactoring Example 1 (The "Do-It-All" class) into Example 2 (The Orchestrator) to demonstrate how the original `ModelPipeline` code was smelly.

It needed to be refactored because the data source, data cleaning and modelling were subject to changes for various different reasons, and in its current state the code was too brittle to maintain effectively.

#### Example 1

```python
class ModelPipeline:
    def __init__(self, data_path):
        self.data_path = data_path

    def load_from_s3(self):
        print(f"Connecting to S3 to get {self.data_path}")
        return "raw_data"

    def clean_txn_data(self, data):
        print("Cleaning specific transaction JSON format")
        return "cleaned_data"

    def train_xgboost(self, data):
        print("Running XGBoost trainer")
        return "model"
```


#### Example 2

```python
class S3DataLoader:
    """Handles only Infrastructure concerns."""
    def __init__(self, data_path):
        self.data_path = data_path

    def load(self):
        print(f"Connecting to S3 to get {self.data_path}")
        return "raw_data"

class TransactionsCleaner:
    """Handles only Data Domain/Schema concerns."""
    def clean(self, data):
        print("Cleaning specific transaction JSON format")
        return "cleaned_data"

class XGBoostTrainer:
    """Handles only ML/Research concerns."""
    def train(self, data):
        print("Running XGBoost trainer")
        return "model"

class ModelPipeline:
    """The Orchestrator: It knows 'what' to do, but not 'how' to do it."""
    def __init__(self, loader, cleaner, trainer):
        self.loader = loader
        self.cleaner = cleaner
        self.trainer = trainer

    def run(self):
        data = self.loader.load()
        cleaned = self.cleaner.clean(data)
        return self.trainer.train(cleaned)
```

Previously, we implicitly assumed that this was production grade code that was subject to the various maintenance changes/feature additions that are frequently made for such code. 

But what if this was a new product/data source/model that we were building out as part of an MVP or R&D? Would the same 'Divergent Change' code-smell apply in this context?

In such a scenario, opting for example 2 may actually be the smellier choice.

If the scope of the project is to consider one data source, or one model, building three separate classes and an orchestrator may count as 'pre-solving' problems you don't yet have.

## The Overarching Lesson

What these two code smells reveal is that software engineering is rarely about "correct" code. It is about context.

A coding agent can write perfect Python in both function and syntax, but it doesn't know your entire business context. It doesn't know if the script it’s writing is a throwaway experiment or the backbone of a multi-million dollar production pipeline revamp.

### Efficiency tradeoffs

You could argue that we can simply feed the AI every little detail of business context, from the meetings you've had to the tea-break chats you had with a fellow colleague. But in practice, that isn't scalable.

If you have to spend two hours writing a "context memo" just to get a clean 50-line function, have you really gained efficiency? Or have you just transformed the manual labor of writing code into that of writing prompts?

### What makes you stand out from the rest

In the age of AI, your value as a data scientist has fundamentally changed. The manual labour of writing code has now been removed. Agents will handle the boilerplating, the formatting, and unit testing.

You are now free to focus more on the problem solving and architecting a solution. Isn't that ultimately the 'fun' of data science?

So, to make yourself stand out from the other data scientists who are blindly copy pasting code, you need to have the structural intuition to guide the coding agent in a direction that is relevant for your unique situation.

That intuition comes from knowing about the code smells we've discussed in this article, and the other two concepts (design patterns, abstraction) that we will delve into in subsequent articles.

## The Power of Abstraction

This "Lego-brick" flexibility doesn't happen by accident. It is powered by one of the most fundamental concepts in software engineering called 'abstraction'.

Notice how in the `ModelPipeline.run` method, you don't see any mention of S3, JSON or XGBoost. It only uses generic temrs like `load`, `clean` and `train`.

```python
def run(self):
    data = self.loader.load()
    cleaned = self.cleaner.clean(data)
    return self.trainer.train(cleaned)
```

In software engineering, Abstraction are generalisations that define key characteristics and behaviour of that piece of code.

A commonly used example is a car's steering wheel. Everyone knows that what it does - turn it left, car goes left. Turn it right, car goes right.

But most drivers have no clue why or how it works like this. And they don't need to! The driver doesn't need to open up the car bonnet and understand every single component to be able to drive.

That's why when electric cars first came to be, everyone could still drive them even though the internals of the car have entirely changed.

Coming back to coding world, abstractions allow us to understand code and communicate how they work with 'concepts' and 'ideas' that are intuitive to us, rather than pointing to line 598 of file `my_god_code_that_does_everything.py` and having a 1 hour call with your colleague about what it's doing and how.

Still don't quite get the concept? Well, let's take a look at the most commonly used abstraction in DS/ML to illustrate its usefulness.

## The most common Abstraction in DS/ML

```python
from my_fake_package.fraud_detection import RomanceScamBinaryClassifier

X_train, y_train, X_test, y_test = load_data()
model = RomanceScamBinaryClassifer()

model.fit(X_train, y_train)

y_score = model.predict_proba(X_test)
```

Now, anyone who has ever done any sort of data science would read the above code and go "ah okay I see what's happening here". Even if you don't have a good idea, you can at least get the gist.

1. We're using some binary classifcation model for detecting Romance scams.
2. We `fit` the model on train data.
3. We generate scores on the test set using `predict_proba`.

There are a **LOT** of subtle abstractions at play here that allows any reader to grasp what is happening here in an instant, all possible because we are familiar with the `scikit-learn` Estimator class.

But to use this classifier, do you need to scour through the `RomanceScamBinaryClassifier.fit` or `predict_proba` methods to know how to use it? 

No.

You read the docstrings/documentation, understand how to use it, and get going. Similarly, when was the last time you've had to go into scikit-learn's github source code and look at the underlying implementation? For most readers, never.